In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler


In [3]:

# Download data
stock = "GOOG"
start = "2012-01-01"
end = "2022-12-31"

data = yf.download(stock, start, end)


[*********************100%***********************]  1 of 1 completed


In [ ]:
data = data[['Close']]
split_index = int(len(data) * 0.8)

data_train = data[:split_index]
data_test = data[split_index:]


In [5]:

scaler = MinMaxScaler(feature_range=(0,1))
data_train_scaled = scaler.fit_transform(data_train)


In [6]:

x = []
y = []

for i in range(100, data_train_scaled.shape[0]):
    x.append(data_train_scaled[i-100:i])
    y.append(data_train_scaled[i,0])

x = np.array(x)
y = np.array(y)

x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).view(-1,1)


In [7]:

# LSTM Model
class StockLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm1 = nn.LSTM(1,50,batch_first=True)
        self.dropout1 = nn.Dropout(0.2)

        self.lstm2 = nn.LSTM(50,60,batch_first=True)
        self.dropout2 = nn.Dropout(0.3)

        self.lstm3 = nn.LSTM(60,80,batch_first=True)
        self.dropout3 = nn.Dropout(0.4)

        self.lstm4 = nn.LSTM(80,120,batch_first=True)
        self.dropout4 = nn.Dropout(0.5)

        self.fc = nn.Linear(120,1)

    def forward(self,x):

        x,_ = self.lstm1(x)
        x = self.dropout1(x)

        x,_ = self.lstm2(x)
        x = self.dropout2(x)

        x,_ = self.lstm3(x)
        x = self.dropout3(x)

        x,_ = self.lstm4(x)
        x = self.dropout4(x)

        x = x[:,-1,:]
        x = self.fc(x)

        return x




In [8]:
model = StockLSTM()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
batch_size = 32

for epoch in range(epochs):

    for i in range(0,len(x),batch_size):

        batch_x = x[i:i+batch_size]
        batch_y = y[i:i+batch_size]

        optimizer.zero_grad()

        outputs = model(batch_x)

        loss = criterion(outputs,batch_y)

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs} Loss: {loss.item()}")

torch.save(model.state_dict(),"stock_model.pth")

print("Model trained and saved as stock_model.pth")

Epoch 1/20 Loss: 0.01883169636130333
Epoch 2/20 Loss: 0.011904501356184483
Epoch 3/20 Loss: 0.06016223132610321
Epoch 4/20 Loss: 0.2020585834980011
Epoch 5/20 Loss: 0.1725459098815918
Epoch 6/20 Loss: 0.004190279170870781
Epoch 7/20 Loss: 0.008597147651016712
Epoch 8/20 Loss: 0.2605544328689575
Epoch 9/20 Loss: 0.23006120324134827
Epoch 10/20 Loss: 0.22967727482318878
Epoch 11/20 Loss: 0.254644513130188
Epoch 12/20 Loss: 0.25698864459991455
Epoch 13/20 Loss: 0.2210017293691635
Epoch 14/20 Loss: 0.21246403455734253
Epoch 15/20 Loss: 0.22427786886692047
Epoch 16/20 Loss: 0.22641903162002563
Epoch 17/20 Loss: 0.22121655941009521
Epoch 18/20 Loss: 0.17454218864440918
Epoch 19/20 Loss: 0.1980307400226593
Epoch 20/20 Loss: 0.22252662479877472
Model trained and saved as stock_model.pth
